# Week 13 作業：DeepFace 人臉比對

# **建立一個blackpink分辨器！**

In [1]:
# blackpink英文名, 這是因為資料夾名稱用中文有時會出問題
members_en = "Jisoo,Jennie,Lisa,Rose"

# 成員中文名, 顯示時用的名稱
members_zh = "Jisoo,Jennie,Lisa,Rose"

# APP 的名稱
title = "blackpink PK 戰"

# APP 的說明
description="和 AI PK blackpink的辨識能力"

因為使用github去連結檔案，所以在這邊把它註解

In [2]:
# from google.colab import drive
# drive.mount('/content/drive')

### 1. 安裝 DeepFace 套件

In [3]:
!pip install gradio
!pip install deepface

In [4]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#用於處理 zip 格式的壓縮文件。
import zipfile

接下來讀入需要的套件。

In [5]:
from deepface import DeepFace
from PIL import Image
import gradio as gr
import os

### 2. 創造資料夾

以成員名字分開建立資料夾, 並用逗號隔開。

In [6]:
# name 這個字典就是用來對應英文和中文名字的

members = members_en.split(',')
name_zh = members_zh.split(',')
member_name = dict(zip(members, name_zh))
N = len(members)

以下是關於github家縮資料夾的串接

In [7]:
photo_folder = "/content/photos/"

In [8]:
# prompt: 建立 photos 資料夾, 並為上面 members 串列中毎個都建一個資料夾。

for member in members:
    os.makedirs(f'./photos/{member}', exist_ok=True)

> 原本的資料集 `Yunyuhu/blackpink` 目前設為私人，下面的下載會失敗。想執行的話，可以改放自己準備的照片：每位成員一個子資料夾，各放至少一張照片。

In [9]:
!wget --no-check-certificate \
    https://github.com/Yunyuhu/blackpink/raw/main/blackpink.zip\
    -O /content/photos/blackpink.zip

--2024-05-29 02:35:53--  https://github.com/Yunyuhu/blackpink/raw/main/blackpink.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Yunyuhu/blackpink/main/blackpink.zip [following]
--2024-05-29 02:35:53--  https://raw.githubusercontent.com/Yunyuhu/blackpink/main/blackpink.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8820963 (8.4M) [application/zip]
Saving to: ‘/content/photos/blackpink.zip’

/content/photos/bla 100%[===================>]   8.41M  --.-KB/s    in 0.1s    

2024-05-29 02:35:53 (60.9 MB/s) - ‘/content/photos/blackpink.zip’ saved [8820963/8820963]



In [10]:
local_zip = '/content/photos/blackpink.zip'
zip_ref = zipfile.ZipFile(local_zip, 'r')
# 排除__MACOSX資料夾
file_list = [file for file in zip_ref.namelist() if '__MACOSX' not in file]
zip_ref.extractall('/content/photos', members=file_list)
zip_ref.close()

確認資料是否串接正確，列出crack的資料夾照片確認資料。

In [11]:
thedir = photo_folder + members[0]
os.listdir(thedir)

['Jisoo02.jpeg',
 '.ipynb_checkpoints',
 '.DS_Store',
 'Jisoo01.jpeg',
 'Jisoo03.jpeg']

### *** 請開始在每個成員的資料夾下，放入至少一張該成員的照片 ***

接著就可以一路執行到最後!

### 3. 用 Gradio 打造 PK 程式

In [12]:
# 用 DeepFace 辨識是哪為成員
def rec(img):
    if img.lower().endswith(('.jpg', '.jpeg', '.png', '.gif')):
        try:
            df = DeepFace.find(img, db_path=photo_folder, enforce_detection=False)
            result = df[0].identity[0].split('/')[-2]
        except:
            result = "未知"
        return result
    else:
        return "Invalid image file"


# 回傳中文名稱

def pk(img, name):
    result = member_name[rec(img)]
    return result

### 5. 用 Gradio 打造 PK 程式

In [13]:
inp1 = gr.Image(label="QUIZ", type="filepath")
inp2 = gr.Dropdown(
            name_zh,
            label="你的答案:"
        )
out = gr.Text(label="我的答案:")

In [ ]:
iface = gr.Interface(pk,
                     inputs=[inp1, inp2],
                     outputs=out,
                     title=title,
                     description=description)

iface.launch(share=True, debug=True)